# 02. Advanced Modeling & Stacking

This notebook trains an XGBoost model, a K-Nearest Neighbors (KNN) model, and finally combines them using a Stacking Ensemble approach (with Logistic Regression as the meta-learner) to maximize predictive accuracy.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')

from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_score

from src.data_processing import load_and_clean_data, get_train_test_data
from src.evaluation import print_model_performance, plot_confusion_matrix

## Data Preparation

In [ ]:
np.random.seed(87)
df = pd.DataFrame({
    'Glucose': np.random.normal(120, 30, 500),
    'BloodPressure': np.random.normal(70, 10, 500),
    'Insulin': np.random.exponential(50, 500),
    'Age': np.random.randint(21, 80, 500),
    'Outcome': np.random.randint(0, 2, 500)  # Binary classification
})

X_train, X_test, y_train, y_test = get_train_test_data(df, target_col='Outcome')
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## Train Base Models: XGBoost & KNN

In [ ]:
# 1. XGBoost
xgb_model = XGBClassifier(max_depth=10, learning_rate=0.1, n_estimators=100,
                          reg_alpha=0.005, subsample=0.8,
                          gamma=0, objective='binary:logistic', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print_model_performance(y_test, xgb_pred, model_name="XGBoost")

# 2. KNN
knn_model = KNeighborsClassifier(n_neighbors=9)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
print_model_performance(y_test, knn_pred, model_name="KNN")

## Model Stacking
Combining the strengths of XGBoost (Tree/Gradient based) and KNN (Distance based) using a Logistic Regression meta-model.

In [ ]:
base_models = [
    ('xgboost', xgb_model),
    ('knn', knn_model)
]

meta_model = LogisticRegression()
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

stacking_clf.fit(X_train, y_train)
stacking_pred = stacking_clf.predict(X_test)

print_model_performance(y_test, stacking_pred, model_name="Stacking Ensemble")
plot_confusion_matrix(y_test, stacking_pred, title="Stacking Confusion Matrix")